# Credit Scoring with Logistic Regression – Extended Solution

**Domain:** Banking / Credit Risk / Retail Lending  
**Topic:** Logistic Regression Model Representation for Probability of Default  
**Extended with:** vectorized sigmoid, decision threshold, accuracy, more practice, threshold/policy simulation, audience notes, flowchart

![Flowchart](credit_scoring_logistic_flowchart.png)


## Goals
- Implement the sigmoid and the logistic model \( f_{w,b}(x) = \sigma(wx + b) \)
- Apply it to credit scoring (DTI → P(default))
- Convert probabilities into approve/reject decisions via a threshold
- Explore how threshold and parameters affect risk policy


## Notation (Credit Risk context)
| Notation | Description | Python |
|:---------|:------------|:-------|
| \( x \) | Debt-to-Income ratio (scaled) | `x_train` |
| \( y \) | 1 = defaulted, 0 = no default | `y_train` |
| \( w, b \) | model parameters | `w`, `b` |
| \( \sigma(z) \) | sigmoid | `sigmoid` |
| \( f_{w,b}(x) \) | \( P(\text{default} \mid x) \) | `f_wb` |
| threshold | risk cutoff (policy lever) | `threshold` |


## 0. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

print("Libraries imported successfully.")


## 1. Problem Statement – Credit Scoring

A bank wants to estimate the probability that an applicant will default given their Debt-to-Income (DTI) ratio.

Illustrative data (higher DTI → higher default risk):

| DTI (scaled) | Defaulted (1=Yes) |
|--------------|-------------------|
| 0.5          | 0                 |
| 1.0          | 0                 |
| 1.5          | 0                 |
| 2.0          | 1                 |
| 2.5          | 1                 |
| 3.0          | 1                 |


### Create training data


In [ ]:
x_train = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
y_train = np.array([0, 0, 0, 1, 1, 1])
print(f"x_train (DTI) = {x_train}")
print(f"y_train (default) = {y_train}")
print(f"Number of applicants m = {len(x_train)}")


## 2. Visualize Binary Credit Data


In [ ]:
plt.figure(figsize=(7, 4))
# slight jitter so points are visible
rng = np.random.default_rng(42)
y_jitter = y_train + rng.normal(0, 0.03, size=len(y_train))
plt.scatter(x_train, y_jitter, c=y_train, cmap='coolwarm', s=80, edgecolors='k', label='Applicants')
plt.yticks([0, 1], ['No Default (0)', 'Default (1)'])
plt.xlabel('Debt-to-Income Ratio (scaled)')
plt.ylabel('Default Outcome')
plt.title('Credit Data: DTI vs Observed Default')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 3. The Sigmoid Function

\[ \sigma(z) = \frac{1}{1 + e^{-z}} \]


In [ ]:
def sigmoid(z):
    """Compute the sigmoid of z (scalar or array)."""
    return 1 / (1 + np.exp(-z))

print("sigmoid(0)  =", sigmoid(0))
print("sigmoid(10) ≈", sigmoid(10))
print("sigmoid(-10)≈", sigmoid(-10))


In [ ]:
z_vals = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 4))
plt.plot(z_vals, sigmoid(z_vals), 'b-', linewidth=2)
plt.axhline(0.5, color='gray', ls='--', alpha=0.7, label='0.5 threshold')
plt.axvline(0, color='gray', ls='--', alpha=0.5)
plt.title('Sigmoid Function σ(z)')
plt.xlabel('z (logit / linear score)')
plt.ylabel('σ(z) = Probability')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Logistic Model Representation

\[ f_{w,b}(x) = \sigma(wx + b) = P(y=1 \mid x) \]


In [ ]:
def compute_model_output(x, w, b):
    """Return predicted default probabilities."""
    z = w * x + b
    return sigmoid(z)

# Chosen parameters so the curve rises through the data region
w = 2.0
b = -4.0
print(f"Model parameters: w = {w}, b = {b}")
print("Training probabilities:", np.round(compute_model_output(x_train, w, b), 3))


In [ ]:
# Probability curve over a fine grid
x_grid = np.linspace(0, 3.5, 200)
prob_grid = compute_model_output(x_grid, w, b)

plt.figure(figsize=(8, 5))
plt.plot(x_grid, prob_grid, 'b-', linewidth=2, label='P(default | DTI)')
plt.scatter(x_train, y_train, c=y_train, cmap='coolwarm', s=90, edgecolors='k', zorder=5, label='Observed')
plt.axhline(0.5, color='gray', ls='--', alpha=0.8, label='Decision threshold = 0.5')
# decision boundary location: where σ(wx+b) = 0.5 → wx+b = 0 → x = -b/w
x_boundary = -b / w
plt.axvline(x_boundary, color='green', ls=':', alpha=0.8, label=f'Decision boundary x={x_boundary:.2f}')
plt.xlabel('Debt-to-Income Ratio (scaled)')
plt.ylabel('Predicted Probability of Default')
plt.title('Logistic Credit-Scoring Model')
plt.ylim(-0.05, 1.05)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()


## 5. Decision Threshold & Credit Decision


In [ ]:
def predict(x, w, b, threshold=0.5):
    """Return (probabilities, binary decisions)."""
    probs = compute_model_output(x, w, b)
    decisions = (probs >= threshold).astype(int)
    return probs, decisions

probs, decisions = predict(x_train, w, b, threshold=0.5)
print("DTI      :", x_train)
print("P(default):", np.round(probs, 3))
print("Decision :", decisions, "  (1 = high risk / potential reject)")
print("Actual y :", y_train)


## 6. Prediction for New Applicants


In [ ]:
new_dti = np.array([0.8, 1.8, 2.7])
new_probs, new_dec = predict(new_dti, w, b, threshold=0.5)
for dti, p, d in zip(new_dti, new_probs, new_dec):
    risk = "HIGH RISK (reject / price up)" if d == 1 else "LOW RISK (approve)"
    print(f"DTI = {dti:.1f}  →  P(default) = {p:.3f}  →  {risk}")


## 7. Alternate Implementation – Fully Vectorized + scipy


In [ ]:
def sigmoid_vectorized(z):
    return 1.0 / (1.0 + np.exp(-z))

def compute_model_output_vectorized(x, w, b):
    return sigmoid_vectorized(w * x + b)

# Equivalence check
print("Manual  :", np.round(compute_model_output(x_train, w, b), 4))
print("Vector  :", np.round(compute_model_output_vectorized(x_train, w, b), 4))

# Optional: scipy
try:
    from scipy.special import expit
    print("scipy   :", np.round(expit(w * x_train + b), 4))
except ImportError:
    print("scipy.special.expit not available – pure NumPy is sufficient.")


## 8. More Practice Exercises


In [ ]:
# Practice 1 – Effect of different thresholds on number of high-risk flags
print("Threshold sensitivity on training set:")
for thr in [0.3, 0.5, 0.7]:
    _, dec = predict(x_train, w, b, threshold=thr)
    n_high = np.sum(dec)
    print(f"  threshold = {thr:.1f}  →  {n_high} / {len(x_train)} applicants flagged high-risk")

# Practice 2 – Steeper vs flatter sigmoid (change w)
print("\nEffect of slope w (keeping decision boundary roughly similar):")
for w_alt in [1.0, 2.0, 4.0]:
    b_alt = -w_alt * 2.0   # keep boundary near x=2
    p = compute_model_output(x_train, w_alt, b_alt)
    print(f"  w={w_alt:.1f}  probs = {np.round(p, 3)}")

# Practice 3 – Simple training accuracy at threshold 0.5
_, dec = predict(x_train, w, b, threshold=0.5)
accuracy = np.mean(dec == y_train)
print(f"\nTraining accuracy (threshold=0.5): {accuracy:.1%}")


## 9. Simulation Section – Credit Policy Levers

Edit the parameters and re-run. Observe:
- How the probability curve moves
- Where the decision boundary sits
- How many applicants would be declined under the chosen threshold


In [ ]:
# ========== SIMULATION PARAMETERS (edit freely) ==========
sim_w         = 2.0
sim_b         = -4.0
sim_threshold = 0.5
# ========================================================

x_grid = np.linspace(0, 3.5, 300)
prob_grid = compute_model_output(x_grid, sim_w, sim_b)
x_boundary = -sim_b / sim_w if sim_w != 0 else np.nan

probs, decisions = predict(x_train, sim_w, sim_b, threshold=sim_threshold)
n_high = np.sum(decisions)
approval_rate = 1 - n_high / len(x_train)

print(f"Parameters: w={sim_w}, b={sim_b}, threshold={sim_threshold}")
print(f"Decision boundary (P=0.5) at DTI ≈ {x_boundary:.2f}")
print(f"High-risk flags on training set: {n_high}/{len(x_train)}")
print(f"Implied approval rate (if high-risk = reject): {approval_rate:.0%}")

plt.figure(figsize=(8, 5))
plt.plot(x_grid, prob_grid, 'b-', lw=2, label='P(default | DTI)')
plt.scatter(x_train, y_train, c=y_train, cmap='coolwarm', s=90, edgecolors='k', zorder=5)
plt.axhline(sim_threshold, color='red', ls='--', alpha=0.8, label=f'Threshold = {sim_threshold}')
plt.axvline(x_boundary, color='green', ls=':', alpha=0.8, label=f'Boundary ≈ {x_boundary:.2f}')
plt.xlabel('Debt-to-Income Ratio (scaled)')
plt.ylabel('Predicted P(default)')
plt.title('Credit Policy Simulation – Change w, b or threshold')
plt.ylim(-0.05, 1.05)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()


## 10. Audience Adaptation Notes (Credit Risk stakeholders)

**Primary audience – Credit Risk analysts / model developers**  
→ Show the sigmoid, the probability curve, residual/accuracy checks, and the effect of changing w and the threshold. They care about discrimination and calibration.

**Secondary audience – Credit Committee / senior management**  
→ Emphasize the single policy question: “At this threshold, what share of applicants would we decline?” and the location of the decision boundary on the DTI axis. One clean chart + the approval-rate number is usually enough.

**Regulatory / audit note**  
In real portfolios the parameters are estimated under strict governance; this notebook is pedagogical only.


## Congratulations!
You now have a working logistic-regression model representation for credit scoring:
- Sigmoid turns a linear credit score into a probability of default
- A threshold implements the bank’s risk appetite
- Changing w, b or the threshold is a direct way to explore alternative credit policies

This is the foundation for full scorecard development, PD model estimation, and regulatory capital calculations.
